# 5.21 — Creating a Baseline Model Using Simple Heuristics

A baseline answers one question:

> **Is my model better than a trivial solution?**

This notebook builds two baselines for a binary classification problem:

1. **Majority-class baseline** using `DummyClassifier(strategy="most_frequent")`
2. **Simple heuristic baseline** using a single numeric threshold derived from the **training** set only

Then we compare both against this repo’s Logistic Regression model.

Leakage rule:

- Split first
- Derive any heuristic thresholds using **training data only**

In [ ]:
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

from src.data_loader import load_csv
from src.feature_engineering import engineer_features
from src.model import train_model
from src.evaluate import evaluate_model
from src.preprocessing import (
    separate_features_and_target,
    split_data,
    fit_preprocessor,
    transform_features,
)
from src.config import (
    ALL_FEATURES,
    CATEGORICAL_FEATURES,
    EXCLUDED_COLUMNS,
    NUMERICAL_FEATURES,
    TARGET_COLUMN,
    TARGET_SOURCE_COLUMN,
)

## 1) Load data + define target (demo)

For the included demo dataset, we derive a binary target from `yield_kg` (high vs low relative to median).

`yield_kg` is **not** allowed as a feature.

In [ ]:
df = load_csv("data/raw/source_demo_crops_20260321.csv")

working = df.copy()
working[TARGET_COLUMN] = (working[TARGET_SOURCE_COLUMN] >= working[TARGET_SOURCE_COLUMN].median()).astype(int)

print("Target distribution:")
print(working[TARGET_COLUMN].value_counts(normalize=True))
working.head()

## 2) Separate features and target using explicit feature lists

This is the repo’s discipline step:

- numeric features and categorical features are explicitly listed
- excluded columns are validated

In [ ]:
X, y = separate_features_and_target(
    working,
    target_column=TARGET_COLUMN,
    feature_columns=[c for c in ALL_FEATURES if c in working.columns],
    excluded_columns=EXCLUDED_COLUMNS,
)

X = engineer_features(X)

X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2, random_state=42)

print("Train size:", X_train.shape, " Test size:", X_test.shape)

## 3) Baseline #1 — Majority-class (`most_frequent`)

This baseline ignores features entirely and predicts the most common class in `y_train`.

In [ ]:
# Use the same preprocessing as the real model so the comparison is apples-to-apples.
# (DummyClassifier itself ignores X, but it expects X to be passed.)
bundle = fit_preprocessor(
    X_train,
    numeric_features=NUMERICAL_FEATURES,
    categorical_features=CATEGORICAL_FEATURES,
)
X_train_t = transform_features(X_train, bundle)
X_test_t = transform_features(X_test, bundle)

baseline_majority = DummyClassifier(strategy="most_frequent")
baseline_majority.fit(X_train_t, y_train)

baseline_majority_metrics = evaluate_model(baseline_majority, X_test_t, y_test)

print("Majority baseline:")
print(f"  accuracy={baseline_majority_metrics['accuracy']:.4f}")
print(f"  f1={baseline_majority_metrics['f1_score']:.4f}")

## 4) Baseline #2 — Simple heuristic (single-threshold rule)

We define a trivial rule using **training data only**.

Example rule (demo):

- predict class 1 if `price` is above the **training** median

This shows how a business rule baseline can be implemented safely.

In [ ]:
if "price" not in X_train.columns:
    raise ValueError("This demo heuristic expects a 'price' feature")

threshold = float(X_train["price"].median())  # derived from TRAIN ONLY

y_pred_rule = (X_test["price"] >= threshold).astype(int)

precision, recall, f1, _ = precision_recall_fscore_support(
    y_test,
    y_pred_rule,
    average="binary" if y_test.nunique() == 2 else "weighted",
    zero_division=0,
)

rule_metrics = {
    "accuracy": accuracy_score(y_test, y_pred_rule),
    "precision": precision,
    "recall": recall,
    "f1_score": f1,
}

print("Heuristic baseline (price >= train_median):")
print(f"  threshold={threshold}")
print(f"  accuracy={rule_metrics['accuracy']:.4f}")
print(f"  f1={rule_metrics['f1_score']:.4f}")

## 5) Real model — Logistic Regression

Now we train the real model and compare against baselines.

In [ ]:
model = train_model(X_train_t, y_train)
model_metrics = evaluate_model(model, X_test_t, y_test)

print("Logistic Regression:")
print(f"  accuracy={model_metrics['accuracy']:.4f}")
print(f"  f1={model_metrics['f1_score']:.4f}")

## 6) Side-by-side comparison

A model result without a baseline is easy to misread.

You want to report improvements like:

- $\Delta$ accuracy = model accuracy − baseline accuracy
- $\Delta$ F1 = model F1 − baseline F1

In [ ]:
comparison = pd.DataFrame(
    [
        {"name": "baseline_majority", "accuracy": baseline_majority_metrics["accuracy"], "f1": baseline_majority_metrics["f1_score"]},
        {"name": "baseline_rule", "accuracy": rule_metrics["accuracy"], "f1": rule_metrics["f1_score"]},
        {"name": "logreg", "accuracy": model_metrics["accuracy"], "f1": model_metrics["f1_score"]},
    ]
).set_index("name")

comparison["delta_accuracy_vs_majority"] = comparison["accuracy"] - float(baseline_majority_metrics["accuracy"])
comparison["delta_f1_vs_majority"] = comparison["f1"] - float(baseline_majority_metrics["f1_score"])
comparison

## Takeaways

- Always establish at least a majority-class baseline for classification.
- A rule-based baseline can be valuable, but derive thresholds from **training data only**.
- If your model can’t beat the baseline on the right metric (often F1 for imbalanced data), improve features and data before tuning models.